In [ ]:
from io import StringIO
import glob
import numpy as np
import pandas as pd

STANDARD_COLUMNS = [ # 표준 22개 컬럼 순서 지정
    "Time",
    "dt",
    "Roll",
    "Pitch",
    "AccX",
    "AccY",
    "AccZ",
    "GyroX",
    "GyroY",
    "GyroZ",
    "ThetaMag",
    "Length",
    "L_dot",
    "L_dot_cmd",
    "L_enc_meas",
    "L_dot_encoder",
    "MotorCmd",
    "Error",
    "S_val",
    "SMCState",
    "TinyMLComp",
    "RawEnc",
]

file_paths = glob.glob("./data_2_controlled_0820/*.csv") #학습 데이터 저장 경로
df_list = [] #모아둘 빈 리스트

for f in file_paths:
    with open(f, "r", encoding="utf-8", errors="ignore") as file: #읽기 모드로 열기
        
        filtered_lines = [ # #으로 시작하는 CONFIG 행 거르기
            line for line in file if not line.strip().startswith("#")
        ]

    if not filtered_lines:
        continue

    has_header = filtered_lines[0].strip().startswith("Time")  # 첫 번째 줄 헤더 포함 확인

    if has_header: # 헤더가 있는 파일: 기존 헤더 사용
        temp_df = pd.read_csv(StringIO("".join(filtered_lines))) #하나의 문자열로 합친 후 Pandas Dataframe으로 읽어오기
        temp_df.columns = temp_df.columns.str.strip()
    else: # 헤더가 없는 파일: 강제로 STANDARD_COLUMNS 이름을 부여하여 파싱
        temp_df = pd.read_csv(
            StringIO("".join(filtered_lines)),
            header=None,
            names=STANDARD_COLUMNS, #전에 정의한 이름 부여함
        )

    temp_df["L_dot_cmd"] = pd.to_numeric( 
        temp_df["L_dot_cmd"], errors="coerce" # L_dot_cmd(속도 지령)을 숫자로 변환
    )

    # 속도 지령이 0이 아닌(모터 제어가 수행된) 인덱스 찾기
    active_indices = temp_df.index[temp_df["L_dot_cmd"] != 0].tolist()

    if active_indices:
        # 1. 시작점: 모터 동작 약 0.5초 전부터 포함 (음수는 안 되도록 최소 0으로 제한)
        start_idx = max(0, active_indices[0] - 50)

        # 2. 종료점: 모터 정지 후  2.5초(250행) 여유분 추가 (과도 응답 학습)
        end_idx = min(len(temp_df) - 1, active_indices[-1] + 250)

        temp_df = temp_df.iloc[start_idx : end_idx + 1].copy() # 해당 구간만 슬라이싱

        df_list.append(temp_df) # 정상 슬라이싱된 데이터만 통합 리스트에 추가
    else:
        # 모터 동작 기록이 없는 파일은 건너뜀 
        print(f"경고: 모터 동작 구획을 찾을 수 없어 제외된 파일: {f}")

df_raw = pd.concat(df_list, ignore_index=True) # 모든 파일 하나로 통합
print(f"1. 전체 원본 데이터(유효 구간 추출 완료): {len(df_raw)} 행")

target_cols = [ # 학습에 쓸 12개 컬럼 선택 
    "AccX",
    "AccY",
    "AccZ",
    "GyroX",
    "GyroY",
    "GyroZ",
    "Roll",
    "Pitch",
    "ThetaMag",
    "Length",
    "L_dot",
    "S_val",
]

for col in target_cols: # 숫자형으로 강제 변환
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

# 결측치(NaN) 정제
df_valid = df_raw.dropna(subset=target_cols).reset_index(drop=True)
print(f"2. 헤더 복구 및 결측치 정제 후 남은 순수 데이터: {len(df_valid)} 행")

df_clean = df_valid[ # 비정상적인 센서 덤프/통신 오류 제거 
    ~(
        (df_valid["AccX"] == 0)
        & (df_valid["AccY"] == 0)
        & (df_valid["AccZ"] == 0)
    )
].copy()

# 3축 가속도로 전체 가속도의 합성 벡터 크기 계산 
acc_magnitude = np.sqrt(
    df_clean["AccX"] ** 2 + df_clean["AccY"] ** 2 + df_clean["AccZ"] ** 2
)

# 충돌 등으로 발생한 스파이크 제거 (조건: 가속도 4.5G 이상, GryoX 회전 속도 500deg/s 이상)
spike_condition = (acc_magnitude < 4.5) & (np.abs(df_clean["GyroX"]) < 500)
df_final = df_clean[spike_condition].copy() 

print(f"3. 최종 충돌 노이즈 정제 후 남은 데이터: {len(df_final)} 행")

1. 전체 원본 데이터(유효 구간 추출 완료): 367050 행
2. 헤더 복구 및 결측치 정제 후 남은 순수 데이터: 367050 행
3. 최종 충돌 노이즈 정제 후 남은 데이터: 364695 행


In [ ]:
from sklearn.model_selection import train_test_split #Train/Test 나눠주는 함수
from sklearn.preprocessing import StandardScaler #정규화 해주는 클래스

feature_cols = [ # X: 입력 변수 (Features 11개)
    "Roll",
    "Pitch",
    "AccX",
    "AccY",
    "AccZ",
    "GyroX",
    "GyroY",
    "GyroZ",
    "ThetaMag",
    "Length",
    "L_dot",
]

X = df_final[feature_cols].values
# y 타겟: 오차를 상쇄하기 위한 역방향 보정량 (-1*S_val)
y = -1.0 * df_final["S_val"].values

# Train(80%) / Test(20%) 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42 #난수 Seed 고정
)

# Z-score 정규화 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) 
X_test_scaled = scaler.transform(X_test) #Data Leakage 방지: 평균, 표준편차 그대로 적용

# 스케일러 계수 (ESP32 main.cpp에 입력)
print("=== [ESP32 main.cpp용 정규화 계수] ===")
print("Scaler Mean :", [round(x, 6) for x in scaler.mean_])
print("Scaler Scale:", [round(x, 6) for x in scaler.scale_])

=== [ESP32 main.cpp용 정규화 계수] ===
Scaler Mean : [np.float64(-0.006287), np.float64(0.01648), np.float64(0.074093), np.float64(-0.078672), np.float64(1.042074), np.float64(0.026497), np.float64(-0.005405), np.float64(-0.028943), np.float64(0.078355), np.float64(0.378174), np.float64(-5e-06)]
Scaler Scale: [np.float64(0.07439), np.float64(0.069916), np.float64(0.046384), np.float64(0.050677), np.float64(0.031026), np.float64(0.461698), np.float64(0.418759), np.float64(1.938547), np.float64(0.067777), np.float64(0.260075), np.float64(0.014837)]


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# ESP32 온디바이스용 경량 MLP 모델 구조 정의
model = models.Sequential( #3개의 Dese Layer
    [
        layers.Input(shape=(len(feature_cols),)), #입력층 (11 크기의 1차원 벡터 받음)
        layers.Dense(16, activation="relu"),      #은닉층1 (노드 16개, 활성화 함수로 ReLU 사용)
        layers.Dense(12, activation="relu"),      #은닉층2 (노드 12개, 활성화 함수로 ReLU 사용)
        layers.Dense(1, activation="linear"),     #출력층 (1개의 외란 보정량 출력을 위한 단일 노드)
    ]
)

model.compile(optimizer="adam", loss="mse", metrics=["mae"]) 
#최적화 알고리즘: 경사하강법 (adam)
#손실함수: 평균제곱오차 (mse) (제곱으로 오차에 큰 패널티로 최적화)
#평가지표: 평균절대오차 (mae) (직관적인 오차 크기)
print("TinyML 모델 학습 시작...")
history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=60,
    batch_size=32,
    verbose=1,
)

TinyML 모델 학습 시작...
Epoch 1/60
9118/9118 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0016 - mae: 0.0175 - val_loss: 2.7660e-05 - val_mae: 0.0038
Epoch 2/60
9118/9118 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - loss: 1.3764e-05 - mae: 0.0025 - val_loss: 7.3333e-06 - val_mae: 0.0019
Epoch 3/60
9118/9118 ━━━━━━━━━━━━━━━━━━━━ 13s 1ms/step - loss: 6.3050e-06 - mae: 0.0017 - val_loss: 3.0280e-06 - val_mae: 0.0012
Epoch 4/60
9118/9118 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - loss: 5.0371e-06 - mae: 0.0016 - val_loss: 8.9267e-06 - val_mae: 0.0021
Epoch 5/60
9118/9118 ━━━━━━━━━━━━━━━━━━━━ 14s 1ms/step - loss: 4.6355e-06 - mae: 0.0015 - val_loss: 2.7072e-06 - val_mae: 0.0012
Epoch 6/60
9118/9118 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - loss: 4.5850e-06 - mae: 0.0015 - val_loss: 4.0236e-06 - val_mae: 0.0015
Epoch 7/60
9118/9118 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - loss: 4.2907e-06 - mae: 0.0015 - val_loss: 3.3166e-06 - val_mae: 0.0013
Epoch 8/60
9118/9118 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - loss: 4.0253e-06 - mae: 0

In [ ]:
# INT8 양자화용 Representative Dataset 생성기
def representative_data_gen(): #동적 min/Max 측정 위한 대표 데이터 생성함수
    for i in range(100):
        sample = np.expand_dims(X_train_scaled[i], axis=0).astype(np.float32)
        #i번째 샘플에 배치 차원 추가 후 float32로 지정
        yield [sample]


# TFLite INT8 양자화 변환
converter = tf.lite.TFLiteConverter.from_keras_model(model) #Keras 모델로부터 TFLite 변환기 객체 생성
converter.optimizations = [tf.lite.Optimize.DEFAULT] #기본 TFLite 최적화 기법 (용량 축소, 추론속도 향상)
converter.representative_dataset = representative_data_gen #Representative Dataset 생성기 등록
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8] #NT8 양자화 강제
converter.inference_input_type = tf.int8 #모델의 입력 타입을 int8로 지정
converter.inference_output_type = tf.int8 #모델의 출력을 int8로 지정

tflite_quant_model = converter.convert()

# ESP32에서 불러올 C++ byte array 형태 .h 파일 저장
hex_lines = [f"0x{b:02x}" for b in tflite_quant_model]
c_array = ( #ESP32 플래시 메모리 (PROGMEM)를 위한 C++ 구조
    f"const unsigned char g_wind_model_data[] = {{\n  "
    + ", ".join(hex_lines)
    + f"\n}};\nconst int g_wind_model_data_len = {len(tflite_quant_model)};"
)

with open("wind_model_data.h", "w") as f: #헤더 파일을 쓰기 모드로 열기
    f.write(c_array) #배열코드 문자열은 헤더 파일에 기입

print("\n🎉 C++ 헤더 파일 생성 완료: wind_model_data.h")

INFO:tensorflow:Assets written to: C:\Users\jihok\AppData\Local\Temp\tmpkbpta4m4\assets


INFO:tensorflow:Assets written to: C:\Users\jihok\AppData\Local\Temp\tmpkbpta4m4\assets


Saved artifact at 'C:\Users\jihok\AppData\Local\Temp\tmpkbpta4m4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 11), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1870841139600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1870841140752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1870841138640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1870841138832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1870841139408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1870841139024: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\jihok\miniconda3\envs\tinyml\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



🎉 C++ 헤더 파일 생성 완료: wind_model_data.h
